# Mediterranean Surface Chlorophyll — Monthly Time Series at Two Pixels (1999–2025)

This notebook follows the same structure as
`copernicus_sst_timeseries_graphs.ipynb` but uses chlorophyll instead of
temperature, drawn from the Mediterranean BGC Reanalysis.

| Label | Location | Approx. coordinates |
|---|---|---|
| North Adriatic (Venice) | Northern Adriatic Sea, offshore Venice | 45.3 °N, 12.5 °E |
| Gulf of Lion (near Italy) | Eastern Gulf of Lion, near French-Italian Riviera | 43.3 °N, 7.5 °E |

**Plots produced**
1. Map showing the actual model grid-points selected.
2. Raw chlorophyll monthly time series (1999–2025) for both pixels.
3. Chlorophyll anomaly monthly time series for both pixels (with ±1 σ envelope).
4. Monthly climatology (mean seasonal cycle) for both pixels.
5. Annual mean chlorophyll and annual anomaly bar charts.
6. Per-pixel detailed two-panel figure (raw + anomaly bars).

> **Credentials**: requires a `~/.netrc` entry for `auth.marine.copernicus.eu`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import xarray as xr
import copernicusmarine
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import panel as pn
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pn.extension()

In [ ]:
# Mediterranean bounding box and time range
# BGC reanalysis starts January 1999
MED_BBOX = dict(
    minimum_longitude=-6,
    maximum_longitude=42,
    minimum_latitude=30,
    maximum_latitude=47,
)
TIME_START = "1999-01-01"
TIME_END   = "2025-12-31"

# Pixel definitions — (lat, lon) chosen to lie in open-water model cells
PIXELS = {
    "North Adriatic (Venice)": {"lat": 45.3, "lon": 12.5},
    "Gulf of Lion (near Italy)": {"lat": 43.3, "lon": 7.5},
}
COLORS = {
    "North Adriatic (Venice)": "#1f77b4",   # blue
    "Gulf of Lion (near Italy)": "#d62728",  # red
}

## 1 — Open the chlorophyll dataset (lazy)

The `cmems_mod_med_bgc-plankton_my_4.2km_P1M-m` product provides monthly
mean chlorophyll at 4.2 km resolution. We select the surface layer
(depth nearest 0 m) to obtain a 2-D (time × lat × lon) array.

In [ ]:
%%time
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_bgc-plankton_my_4.2km_P1M-m",
    variables=["chl"],
    start_datetime=TIME_START,
    end_datetime=TIME_END,
    **MED_BBOX,
)

# Surface chlorophyll only (depth nearest 0 m)
chl = ds["chl"].sel(depth=0, method="nearest")
chl.attrs["long_name"] = "Surface Chlorophyll"
chl.attrs["units"]     = "mg m\u207b\u00b3"
print(f"Chlorophyll shape: {dict(zip(chl.dims, chl.shape))}")
chl

## 2 — Long-term climatological mean

In [ ]:
cluster_type = 'Coiled'

In [ ]:
if cluster_type == 'Coiled':
    import coiled
    cluster = coiled.Cluster(
        region="eu-central-1",
        arm=True,   # run on ARM to save energy & cost
        worker_vm_types=["t4g.large"],  # cheap, small ARM instances, 2cpus, 2GB RAM
        n_workers=30, 
        name = "Emma",
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software='protocoast-notebook-arm',  # Conda environment name
        workspace='esip-lab',
        timeout=180   # leave cluster running for 3 min in case we want to use it again
    )

    client = cluster.get_client()

In [ ]:
%%time
chl_clim = chl.mean(dim="time").compute()
chl_clim.attrs["long_name"] = f"Climatological Mean Chlorophyll ({TIME_START[:4]}\u2013{TIME_END[:4]})"
chl_clim.attrs["units"]     = "mg m\u207b\u00b3"
print(f"Clim chlorophyll range: {float(chl_clim.min()):.4f} \u2013 {float(chl_clim.max()):.4f} mg m\u207b\u00b3")

## 3 — Monthly chlorophyll anomaly

In [ ]:
chl_anom = chl - chl_clim
chl_anom.attrs["long_name"] = "Chlorophyll Anomaly"
chl_anom.attrs["units"]     = "mg m\u207b\u00b3"
print(f"Anomaly shape: {dict(zip(chl_anom.dims, chl_anom.shape))}")

## 4 — Extract pixel time series

Select the nearest model grid-point to each target location, then load both
chlorophyll and chlorophyll anomaly into memory (each pixel time series is
tiny: 324 floats).

In [ ]:
pixel_info = {}   # stores actual snapped coordinates
pixel_chl  = {}   # raw chlorophyll time series
pixel_anom = {}   # anomaly time series

for name, coords in PIXELS.items():
    ts_chl  = chl.sel(latitude=coords["lat"],  longitude=coords["lon"],  method="nearest").compute()
    ts_anom = chl_anom.sel(latitude=coords["lat"], longitude=coords["lon"], method="nearest").compute()

    snapped_lat = float(ts_chl.latitude)
    snapped_lon = float(ts_chl.longitude)
    pixel_info[name] = {"lat": snapped_lat, "lon": snapped_lon}
    pixel_chl[name]  = ts_chl
    pixel_anom[name] = ts_anom

    print(f"{name:35s}  requested ({coords['lat']:.1f}\u00b0N, {coords['lon']:.1f}\u00b0E)  "
          f"\u2192  snapped ({snapped_lat:.3f}\u00b0N, {snapped_lon:.3f}\u00b0E)")

## 5 — Location map

Quick sanity check: plot the two selected grid-points on a Mediterranean map.

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
ax.set_extent([-7, 37, 29, 48], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#e8e0d0", zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":", zorder=2)
ax.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.6)

for name, info in pixel_info.items():
    ax.plot(
        info["lon"], info["lat"],
        marker="*", markersize=14,
        color=COLORS[name],
        transform=ccrs.PlateCarree(),
        zorder=3,
        label=f"{name}\n({info['lat']:.3f}\u00b0N, {info['lon']:.3f}\u00b0E)",
    )

ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.set_title("Selected pixel locations", fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Raw chlorophyll monthly time series (1999–2025)

In [ ]:
# Build a tidy pandas DataFrame: one column per pixel
time_index = pd.to_datetime(pixel_chl[list(PIXELS)[0]].time.values)

df_chl = pd.DataFrame(
    {name: ts.values for name, ts in pixel_chl.items()},
    index=time_index,
)
df_chl.index.name = "time"
df_chl.head(3)

In [ ]:
chl_curves = df_chl.hvplot.line(
    x="time",
    y=list(PIXELS.keys()),
    color=list(COLORS.values()),
    line_width=1.2,
    alpha=0.85,
    ylabel="Chlorophyll (mg m\u207b\u00b3)",
    xlabel="",
    title=f"Monthly Surface Chlorophyll \u2014 {TIME_START[:4]}\u2013{TIME_END[:4]}",
    frame_width=900,
    frame_height=350,
    fontscale=1.2,
    legend="top_right",
)
chl_curves

## 7 — Chlorophyll anomaly monthly time series (1999–2025)

In [ ]:
df_anom = pd.DataFrame(
    {name: ts.values for name, ts in pixel_anom.items()},
    index=time_index,
)
df_anom.index.name = "time"
df_anom.head(3)

In [ ]:
anom_curves = df_anom.hvplot.line(
    x="time",
    y=list(PIXELS.keys()),
    color=list(COLORS.values()),
    line_width=1.2,
    alpha=0.85,
    ylabel="Chlorophyll Anomaly (mg m\u207b\u00b3)",
    xlabel="",
    title=f"Monthly Chlorophyll Anomaly (w.r.t. {TIME_START[:4]}\u2013{TIME_END[:4]} mean)",
    frame_width=900,
    frame_height=350,
    fontscale=1.2,
    legend="top_right",
)

# Add a horizontal zero reference line
zero_line = hv.HLine(0).opts(color="black", line_dash="dashed", line_width=0.8, alpha=0.6)

anom_curves * zero_line

## 8 — Monthly climatology (mean seasonal cycle)

Average chlorophyll by calendar month (1–12) to reveal the seasonal bloom
signal at each location.

In [ ]:
MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]

df_cycle_mean = df_chl.groupby(df_chl.index.month).mean()
df_cycle_std  = df_chl.groupby(df_chl.index.month).std()
df_cycle_mean.index = MONTH_NAMES
df_cycle_std.index  = MONTH_NAMES

df_cycle_mean

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for name, color in COLORS.items():
    mean = df_cycle_mean[name]
    std  = df_cycle_std[name]
    ax.plot(MONTH_NAMES, mean, color=color, lw=2, marker="o", ms=5, label=name)
    ax.fill_between(
        MONTH_NAMES,
        mean - std, mean + std,
        color=color, alpha=0.15,
    )

ax.set_ylabel("Chlorophyll (mg m\u207b\u00b3)", fontsize=11)
ax.set_title(
    f"Mean seasonal cycle of Chlorophyll \u00b1 1\u03c3 ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontsize=12,
)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## 9 — Combined view: chlorophyll and anomaly stacked

In [ ]:
(chl_curves + (anom_curves * zero_line)).cols(1)

## 10 — Per-pixel detailed view (matplotlib)

Two-panel figure for each pixel: raw chlorophyll on top, anomaly
(bar-coloured by sign) on the bottom.

In [ ]:
def plot_pixel(name, color):
    ts_chl  = df_chl[name]
    ts_anom = df_anom[name]
    bar_colors = ["#d62728" if v >= 0 else "#1f77b4" for v in ts_anom]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

    # --- top: raw chlorophyll ---
    ax1.plot(ts_chl.index, ts_chl.values, color=color, lw=1.0, alpha=0.9)
    ax1.set_ylabel("Chlorophyll (mg m\u207b\u00b3)", fontsize=10)
    ax1.set_title(name, fontsize=12, fontweight="bold")
    ax1.grid(axis="y", linestyle=":", alpha=0.5)

    # --- bottom: anomaly bars ---
    ax2.bar(ts_anom.index, ts_anom.values, width=25, color=bar_colors, alpha=0.85)
    ax2.axhline(0, color="black", lw=0.8, linestyle="--")
    ax2.set_ylabel("Chlorophyll Anomaly (mg m\u207b\u00b3)", fontsize=10)
    ax2.xaxis.set_major_locator(mdates.YearLocator(5))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax2.grid(axis="y", linestyle=":", alpha=0.5)

    fig.suptitle(
        f"Chlorophyll & Anomaly \u2014 {name} ({TIME_START[:4]}\u2013{TIME_END[:4]})",
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    plt.show()

for name, color in COLORS.items():
    plot_pixel(name, color)

## 11 — Annual chlorophyll and anomaly time series

Resample the monthly pixel time series to yearly means, then plot both raw
chlorophyll and anomaly at annual resolution.

In [ ]:
# Resample monthly DataFrames to annual means
df_chl_annual  = df_chl.resample("YE").mean()
df_anom_annual = df_anom.resample("YE").mean()

# Use integer years as the index for cleaner x-axis labels
df_chl_annual.index  = df_chl_annual.index.year
df_anom_annual.index = df_anom_annual.index.year
df_chl_annual.index.name  = "year"
df_anom_annual.index.name = "year"

df_anom_annual

In [ ]:
# Annual chlorophyll — interactive hvplot
chl_annual_curves = df_chl_annual.hvplot.line(
    x="year",
    y=list(PIXELS.keys()),
    color=list(COLORS.values()),
    line_width=2,
    ylabel="Chlorophyll (mg m\u207b\u00b3)",
    xlabel="Year",
    title=f"Annual Mean Chlorophyll \u2014 {TIME_START[:4]}\u2013{TIME_END[:4]}",
    frame_width=900,
    frame_height=350,
    fontscale=1.2,
    legend="top_right",
)
chl_annual_curves

In [ ]:
# Annual chlorophyll anomaly — bar chart coloured by sign, both pixels side by side
plots = []
for name, color in COLORS.items():
    vals = df_anom_annual[name]
    bar_colors = ["#d62728" if v >= 0 else "#1f77b4" for v in vals]
    p = vals.hvplot.bar(
        color=bar_colors,
        ylabel="Chlorophyll Anomaly (mg m\u207b\u00b3)",
        xlabel="Year",
        title=name,
        frame_width=500,
        frame_height=300,
        fontscale=1.1,
        rot=45,
    ) * hv.HLine(0).opts(color="black", line_dash="dashed", line_width=0.8)
    plots.append(p)

hv.Layout(plots).cols(2)